# SemEval-2020 Task 7: Assessing Humor in Edited News Headlines | Subtask 1: Estimate the funniness of an edited headline on a 0-3 humor scale

**Team 1:** Matthew Vu, Vu Doan, Prapurna Penmathsa, and Amrisha Dwivedi

**Description:** The code in this Jupyter Notebook file performs all the necessary operations to accomplish SemEval-2020 Task 7: Assessing Humor in Edited News Headlines | Subtask 1: Estimate the funniness of an edited headline on a 0-3 humor scale.  Here, we experiment with four 'TFAlbertForSequenceClassification' models (the ALBERT transformed-based model) to predict humor ratings in edited news headlines.  

The first pair of models use exponetial decay to decrease the learning rate over the training process; however, one of the models uses the RMSprop optimizer and the other uses the Adam optimizer.  The second pair of models simply decrease the learning rate by a constant value for every batch during the training process; however, one of the models uses the RMSprop optimizer and the other uses the Adam optimizer.  

The technique of using exponential decay to decrease the learning rate, found in the first pair of models, is unique to our work and not performed in the research conducted by Salih Tuncer (who's work with ALBERT we are basing our code and methods on).  However, the second pair of models that decreased the learning rate by a constant value for every batch during the training process is the same methods used in the research conducted by Salih Tuncer.  In fact, the code for the second pair of models are nearly identical to his code.  

Nonetheless, whichever model performs the best on the validation set is chosen as our best and final model, and its generalization error is calculated against the test set.

**Acknowledgements:** This code and methods are heavily based on the work on of Nabil Hossain (https://github.com/n-hossain/semeval-2020-task-7-humicroedit/tree/master) and Salih Tuncer (https://github.com/SalihTuncer/AssessHumor).

## Step 0: Import libraries and Set Seeds

In [4]:
import pandas as pd;
import numpy as np;
import random;
import sys;
import os;
import tensorflow as tf;
from transformers import AlbertTokenizer;
from tensorflow.keras.optimizers import RMSprop;
from tensorflow.keras.optimizers import Adam;
from tensorflow.keras.losses import MeanSquaredError;
from tensorflow.keras.metrics import RootMeanSquaredError;
from transformers import TFAlbertForSequenceClassification;
from tensorflow.keras.optimizers.schedules import ExponentialDecay;
from tensorflow.keras.models import load_model;

In [5]:
np.random.seed(0)
random.seed(0)
tf.random.set_seed(0)

## Step 1: Baseline RMSE values

The variable we are trying to predict is the mean funniness score of edited news headlines.  That is, the mean funniness score is the average of funniness scores given to an edited news headline by 5 judges (human annotators).

We have a validation set called "dev.csv" that is used to develop models, tune hyperparameters, and choose the optimal model out of a series of possible models.  Also, we have a test set called "test.csv" that is used to evaluate the final, best model to get an estimate on how well the chosen model will perform on unseen data in the future.

So, the first step in our analysis is to compute two baseline root mean square error (RMSE) values.  One baseline RMSE value is equal to the validation set's ("dev.csv") RMSE when the predicted funniness score of all edited news headlines in that validation set ("dev.csv") are equal to the average mean funniness score of all edited news headlines in the training set ("train.csv").  

The other baseline RMSE value is equal to the test set's ("test.csv") RMSE when the predicted funniness score of all edited news headlines in the test set ("test.csv") is equal to the average mean funniness score of all edited news headlines in the training set ("train.csv").

We want our chosen model to perform better than both the validation and test set baseline RMSE values.

The code in this section was adapted from Nabil Hossain (https://github.com/n-hossain/semeval-2020-task-7-humicroedit/tree/master).

In [6]:
# this function reads in the training set and the test/validation set 
# and writes an output file that is a copy of the test/validation set
# with the baseline predictions
# inputs:
# train_loc -> directory of the training set
# test_loc -> directory of the test/validation set
# output_name -> name of the desired output file
def baseline_dataset(train_loc, test_loc, output_name):
    # read in the training and test sets
    train = pd.read_csv(train_loc)    
    test = pd.read_csv(test_loc)
    
    # make predictions = average of meanGrade
    pred = np.mean(train['meanGrade'])    
    test['pred'] = pred
    
    # create and write to output file
    output = test[['id','pred']] 
    out_loc = f'datasets/subtask1/baseline/{output_name}'
    output.to_csv(out_loc, index=False)
    
    # show that the output is written
    print(f'{output_name} successfully written.')

In [7]:
# this function reads in the test/validation set and a dataset containing the baseline 
# predictions to compute the baseline RMSE of that test/validation set
# and returns the baseline RMSE
def baseline_RMSE(truth_loc, prediction_loc):
    # read in the true values
    truth = pd.read_csv(truth_loc, usecols=['id','meanGrade'])
    # read in the baseline predictions
    pred = pd.read_csv(prediction_loc, usecols=['id','pred'])
    
    assert(sorted(truth.id) == sorted(pred.id)),"ID mismatch between ground truth and prediction!"
    
    # compute the baseline RMSE
    data = pd.merge(truth,pred)
    rmse = np.sqrt(np.mean((data['meanGrade'] - data['pred'])**2))
    
    # return baseline RMSE
    return rmse

In [7]:
# create validation set baseline dataset
baseline_dataset('datasets/subtask1/train.csv', 'datasets/subtask1/dev.csv', 'dev_baseline.csv')

dev_baseline.csv successfully written.


In [8]:
# compute the output the validation set RMSE value
validation_baseline = baseline_RMSE('datasets/subtask1/dev.csv', 'datasets/subtask1/baseline/dev_baseline.csv')
print(f'Baseline validation set RMSE = {validation_baseline:.3f}')

Baseline validation set RMSE = 0.578


In [9]:
# create test set baseline dataset
baseline_dataset('datasets/subtask1/train.csv', 'datasets/subtask1/test.csv', 'test_baseline.csv')

test_baseline.csv successfully written.


In [10]:
# compute the output the test set RMSE value
test_baseline = baseline_RMSE('datasets/subtask1/test.csv', 'datasets/subtask1/baseline/test_baseline.csv')
print(f'Baseline test set RMSE = {test_baseline:.3f}')

Baseline test set RMSE = 0.575


We want our final model to get a better baseline validation and test set RMSE of 0.578 and 0.575, respectively.

## Step 2: Data Processing Code

The code presented in all subsequently steps/sections are adapted from Salih Tuncer (https://github.com/SalihTuncer/AssessHumor).

In [8]:
# declare a class called Preprocess that contains all the methods to process the datasets
class Preprocess:

    # initialize the Class
    # inputs:
    # train_path -> the directory of the training data
    # dev_path -> the directory of the validation data
    # test.csv -> the directory of the test data
    # max_seq_length -> ?
    # batch_size -> ?
    def __init__(self,
                 train_path='datasets/subtask1/train.csv',
                 dev_path='datasets/subtask1/dev.csv',
                 test_path='datasets/subtask1/test.csv',
                 max_seq_length=256, batch_size=16
                 ):
        self._max_seq_length = max_seq_length

        # initialize the pre-trained Albert tokenizer
        self._tokenizer = AlbertTokenizer.from_pretrained('albert-base-v2')
        print('Tokenizer ready...\n')

        # read in the training, validation, and test sets
        train_df = pd.read_csv(train_path)
        dev_df = pd.read_csv(dev_path)
        test_df = pd.read_csv(test_path)
        print('Datasets were read...\n')

        # save the data appropriate as a class property
        # process the training set
        self._train = self._process_corpus(train_df)
        self._train = tf.data.Dataset.from_tensor_slices(self._train).batch(batch_size)
        print('Training-dataset preprocessed and ready...\n')

        # process the validation set
        self._dev = self._process_corpus(dev_df)
        self._dev = tf.data.Dataset.from_tensor_slices(self._dev).batch(batch_size)
        print('Dev-dataset preprocessed and ready...\n')

        # process test set 
        self._test = self._process_corpus(test_df)
        self._test = tf.data.Dataset.from_tensor_slices(self._test).batch(batch_size)
        print('Test-dataset preprocessed and ready...\n')

        print('Preprocessing done.\n')

    # process corpus
    def _process_corpus(self, df: pd.DataFrame) -> ({str: np.ndarray}, np.ndarray):
        # create a 2-D array of zeros with shape (len(df), self._max_seq_length)
        # it will store the tokenized version of each sentence in the dataset
        tokens = np.zeros((len(df), self._max_seq_length), dtype='int32')

        # create a 2-D array of zeros with shape (len(df), self._max_seq_length)
        # it stores the attention mask for each sentence
        attention_mask = np.zeros((len(df), self._max_seq_length), dtype='int32')

        # extract the labels (meanGrade) that the model with be trained to predict
        labels = df['meanGrade'].to_numpy(dtype='float32')

        for i in range(len(df)):
            # reformat the sentence and return additionally the replaced word as a whole sentence
            edited_sentence, sentence = self.reformat_sentence(df['original'][i], df['edit'][i])
            # save the encoded sentences in each row of the matrix
            tokens[i, :] = self.tokenize_sentence(edited_sentence, sentence)

            sentence_size = len(sentence.split(' ')) + 2  # CLS-token at the beginning and SEP-token at the end

            total_sentence_size = (sentence_size * 2) - 1  # second sentence has no CLS-token
            # fill the array with as many ones as we have words
            attention_mask[i, :total_sentence_size] = 1.0

        # return a dictionary containing 
        # inputs_id ->  2D numpy array containing the tokenized version of each sentence in the training dataset
        # attention_mask ->  the attention mask to indicate which tokens in each sentence should be attended to by the model
        # labels -> the target labels
        return {'input_ids': tokens,
                'attention_mask': attention_mask
                }, labels

    @staticmethod
    def reformat_sentence(sentence: str, edit: str) -> (str, str):
        # remove <..../> for the original sentence
        sen = sentence.split('<')
        sen[1] = sen[1].replace('/>', '')
        # and put the edited word in the <..../> and save it as a sentence
        sen[1:2] = sen[1].split(' ', 1)
        edited = sen[0] + edit + sen[2]
        return edited, ''.join(sen)

    # [CLS], ..., [SEP], ..., [SEP] | rest filled with [PAD]
    def tokenize_sentence(self, sentence: str, edited_sentence: str) -> [int]:
        # we want to place the special tokens ourself because we need 1 cls and 2 seps
        tokens = [self._tokenizer.cls_token_id] + self._tokenizer.encode(sentence, add_special_tokens=False) \
                 + [self._tokenizer.sep_token_id] + self._tokenizer.encode(edited_sentence, add_special_tokens=False) \
                 + [self._tokenizer.sep_token_id]
        # now we add the PAD-tokens at the end as filler tokens
        return tokens + [self._tokenizer.pad_token_id] * (self._max_seq_length - len(tokens))

    # get the training set
    def get_train(self) -> tf.data.Dataset:
        return self._train

    # get the validation set
    def get_dev(self) -> tf.data.Dataset:
        return self._dev

    # get the test set
    def get_test(self) -> tf.data.Dataset:
        return self._test

## Step 3: The code to initialize the ALBERT model

In [9]:
# declare the model class
class NN:
    # initialize the class
    def __init__(self):
        self._nn = self._create_nn()

    # returns an instance of TFAlbertforSequenceClassification
    def _create_nn(self) -> TFAlbertForSequenceClassification:
        # create a new instance of TFAlbertForSequenceClassification using a pre-trained
        # ALBERT model called 'albert-base-v2 to predict a regression task with num_labels = 1
        return TFAlbertForSequenceClassification.from_pretrained('albert-base-v2', num_labels=1)

    # returns the ALBERT model
    def get_nn(self):
        return self._nn

## Step 3: Preprocess the training, validation set, and test set

In [10]:
# preprocess the datasets and return a utility class which carries the datasets
max_seq_length = 256
batch_size = 16
prep = Preprocess(max_seq_length=max_seq_length, batch_size=batch_size)

# get the processed training and validation set
train_dataset = prep.get_train()
dev_dataset = prep.get_dev()

Tokenizer ready...

Datasets were read...

Training-dataset preprocessed and ready...

Dev-dataset preprocessed and ready...

Test-dataset preprocessed and ready...

Preprocessing done.



## Step 4: Experiment with exponential decay to reduce the learning rate over time

We will first experiment with exponential decay to decrease the learning rate as the models train as well as using either the RMSprop or Adam optimizers.  Salih Tuncer did not experiment with exponential decay in his own research.  The code below is slightly modified from the code provided by Salih Tuncer.

In [14]:
# Define initial learning rate
initial_learning_rate = 1e-5

# Define decay steps and rate
decay_steps = 603
decay_rate = 0.9

# Create learning rate schedule
lr_schedule = ExponentialDecay(
    initial_learning_rate,
    decay_steps=decay_steps,
    decay_rate=decay_rate)

### Step 4.1: Use the RMSPROP optimizer

In [15]:
# first train the model using the RMSprop optimizer
# create an albert model specialised on regression
model_rmsprop = NN().get_nn()

# use the RMSprop optimizer
optimizer_rmsprop = RMSprop(learning_rate = lr_schedule)

# compile the model with the appropriate optimizer, loss function, and evaluation metric
model_rmsprop.compile(optimizer=optimizer_rmsprop, loss=MeanSquaredError(),
            metrics=[RootMeanSquaredError('root_mean_squared_error')])

print("Fit model witt RMSprop optimizer on training data.")

# train the model on the training set for 3 epochs
# and using the validation set for validation
history_rmsprop = model_rmsprop.fit(train_dataset, validation_data=dev_dataset, epochs=3)
print("RMSprop model successfully trained")

# print('Models successfully saved.')

# Evaluate the models against the validation set
results_rmsprop = model_rmsprop.evaluate(dev_dataset, return_dict=True)

# Print the RMSE
print(f"RMSprop RMSE: {results_rmsprop['root_mean_squared_error']}")

All PyTorch model weights were used when initializing TFAlbertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFAlbertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Fit model witt RMSprop optimizer on training data.
Epoch 1/3
604/604 [==============================] - 7809s 13s/step - loss: 0.3525 - root_mean_squared_error: 0.5937 - val_loss: 0.3264 - val_root_mean_squared_error: 0.5713
Epoch 2/3
604/604 [==============================] - 7435s 12s/step - loss: 0.3146 - root_mean_squared_error: 0.5609 - val_loss: 0.2985 - val_root_mean_squared_error: 0.5464
Epoch 3/3
604/604 [==============================] - 7642s 13s/step - loss: 0.2723 - root_mean_squared_error: 0.5218 - val_loss: 0.3023 - val_root_mean_squared_error: 0.5498
RMSprop model successfully trained
152/152 [==============================] - 742s 5s/step - loss: 0.3023 - root_mean_squared_error: 0.5498
RMSprop RMSE: 0.5497849583625793


In [18]:
# save the RMSprop model to a file
model_rmsprop.save('model_rmsprop', save_format='tf')
print("Done saving!")

INFO:tensorflow:Assets written to: model_rmsprop\assets


INFO:tensorflow:Assets written to: model_rmsprop\assets


Done saving!


### Step 4.2: Use the Adam optimizer

In [20]:
# create an albert model specialised on regression
model_adam = NN().get_nn()

# use the Adam optimizer
optimizer_adam = Adam(learning_rate = lr_schedule)

# compile the model with the appropriate optimizer, loss function, and evaluation metric
model_adam.compile(optimizer=optimizer_adam, loss=MeanSquaredError(),
            metrics=[RootMeanSquaredError('root_mean_squared_error')])

# train the model on the training set for 3 epochs
# and using the validation set for validation
history_adam = model_adam.fit(train_dataset, validation_data=dev_dataset, epochs=3)
print("Adam model successfully trained")

All PyTorch model weights were used when initializing TFAlbertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFAlbertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/3
604/604 [==============================] - 9163s 15s/step - loss: 0.3567 - root_mean_squared_error: 0.5973 - val_loss: 0.3288 - val_root_mean_squared_error: 0.5734
Epoch 2/3
604/604 [==============================] - 9665s 16s/step - loss: 0.3400 - root_mean_squared_error: 0.5831 - val_loss: 0.3082 - val_root_mean_squared_error: 0.5551
Epoch 3/3
604/604 [==============================] - 8337s 14s/step - loss: 0.3008 - root_mean_squared_error: 0.5485 - val_loss: 0.2978 - val_root_mean_squared_error: 0.5457
Adam model successfully trained


In [21]:
# Evaluate the model against the validation set
results_adam = model_adam.evaluate(dev_dataset, return_dict=True)

# Print the RMSE
print(f"Adam RMSE: {results_adam['root_mean_squared_error']}")

152/152 [==============================] - 754s 5s/step - loss: 0.2978 - root_mean_squared_error: 0.5457
Adam RMSE: 0.5457258820533752


In [22]:
# save the Adam model to a file
model_adam.save('model_adam', save_format='tf')
print("Done saving!")

INFO:tensorflow:Assets written to: model_adam\assets


INFO:tensorflow:Assets written to: model_adam\assets


Done saving!


## Step 5: Experiment with original methods outlined in the original paper

Second, we will experiment with decreasing the learning rate by a constant value as the models train as well as using either the RMSprop or Adam optimizers.  Salih Tuncer did experiment with this method in his own research, and the code below is identical to his code.

### Step 5.1: Use RMSprop optimizer

In [7]:
# create an albert model specialised on regression
model_rmsprop_original = NN().get_nn()

All PyTorch model weights were used when initializing TFAlbertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFAlbertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
# use the RMSprop optimizer
optimizer_rmsprop_original = tf.keras.optimizers.legacy.RMSprop(learning_rate = 1e-5, decay = 1e-8)

In [9]:
# compile the model with the appropriate optimizer, loss function, and evaluation metric
model_rmsprop_original.compile(optimizer=optimizer_rmsprop_original, loss=MeanSquaredError(),
            metrics=[RootMeanSquaredError('root_mean_squared_error')])

In [10]:
print("Fit model on training data.")
# train the model on the training set for 3 epochs
# using the validation set for validation
history_rmsprop_original = model_rmsprop_original.fit(train_dataset, validation_data=dev_dataset, epochs=3)
print("Original RMSprop model successfully trained")

Fit model on training data.
Epoch 1/3
604/604 [==============================] - 7467s 12s/step - loss: 0.3510 - root_mean_squared_error: 0.5925 - val_loss: 0.3238 - val_root_mean_squared_error: 0.5690
Epoch 2/3
604/604 [==============================] - 7354s 12s/step - loss: 0.3358 - root_mean_squared_error: 0.5795 - val_loss: 0.3089 - val_root_mean_squared_error: 0.5558
Epoch 3/3
604/604 [==============================] - 8507s 14s/step - loss: 0.2990 - root_mean_squared_error: 0.5468 - val_loss: 0.3042 - val_root_mean_squared_error: 0.5515
Original RMSprop model successfully trained


In [11]:
# Evaluate the models against the validation set
results_rmsprop_original = model_rmsprop_original.evaluate(dev_dataset, return_dict=True)

152/152 [==============================] - 667s 4s/step - loss: 0.3042 - root_mean_squared_error: 0.5515


In [12]:
# Print the RMSE for both models
print(f"Original RMSprop RMSE: {results_rmsprop_original['root_mean_squared_error']}")

Original RMSprop RMSE: 0.5515475273132324


In [13]:
# save the original RMSprop model to a file
model_rmsprop_original.save('model_rmsprop_original', save_format='tf')
print("Done saving!")

INFO:tensorflow:Assets written to: model_rmsprop_original\assets


INFO:tensorflow:Assets written to: model_rmsprop_original\assets


Done saving!


### Step 5.2: Use Adam optimizer

In [7]:
# create an albert model specialised on regression
model_adam_original = NN().get_nn()

All PyTorch model weights were used when initializing TFAlbertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFAlbertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
# use the Adam optimizer
optimizer_adam_original = tf.keras.optimizers.legacy.Adam(learning_rate = 1e-5, decay = 1e-8)

In [9]:
# compile the model with the appropriate optimizer, loss function, and evaluation metric
model_adam_original.compile(optimizer=optimizer_adam_original, loss=MeanSquaredError(),
            metrics=[RootMeanSquaredError('root_mean_squared_error')])

In [12]:
print("Fit model on training data.")
# train the model on the training set for 3 epochs
# using the validation set for validation
history_adam_original = model_adam_original.fit(train_dataset, validation_data=dev_dataset, epochs=3)
print("Adam model successfully trained")

Fit model on training data.
Epoch 1/3
604/604 [==============================] - 8024s 13s/step - loss: 0.3491 - root_mean_squared_error: 0.5909 - val_loss: 0.3194 - val_root_mean_squared_error: 0.5652
Epoch 2/3
604/604 [==============================] - 8030s 13s/step - loss: 0.3102 - root_mean_squared_error: 0.5570 - val_loss: 0.2944 - val_root_mean_squared_error: 0.5426
Epoch 3/3
604/604 [==============================] - 7991s 13s/step - loss: 0.2660 - root_mean_squared_error: 0.5157 - val_loss: 0.2976 - val_root_mean_squared_error: 0.5455
Adam model successfully trained


In [13]:
# Evaluate the models against the validation set
results_adam_original = model_adam_original.evaluate(dev_dataset, return_dict=True)

152/152 [==============================] - 643s 4s/step - loss: 0.2976 - root_mean_squared_error: 0.5455


In [14]:
# Print the RMSE for the Adam model
print(f"Original Adam RMSE: {results_adam_original['root_mean_squared_error']}")

Original Adam RMSE: 0.5454941987991333


In [15]:
# save the original Adam model to a file
model_adam_original.save('model_adam_original', save_format='tf')
print("Done saving!")

INFO:tensorflow:Assets written to: model_adam_original\assets


INFO:tensorflow:Assets written to: model_adam_original\assets


Done saving!


In all, the model that uses exponential decay to decrease the learning rate and the RMSprop optimizer has an RMSE value against the validation set of 0.549784958362579.  

The model that uses exponential decay to decrease the learning rate and the Adam optimizer has an RMSE value against the validaion set of 0.545725882053375.  

The model that simply decreases the learning rate by a constant value and uses the RMSprop optimizer has an RMSE value against the validaion set of 0.551547527313232.  

The model that simply decreases the learning rate by a constant value and uses the Adam optimizer has an RMSE value against the validaion set of 0.545494198799133.

As one can see, the model that performs the best in the one that decreases the learning rate by a constant value and uses the Adam optimizer. This is one of the original methods conducted by Salih Tuncer.

However, it is important to note that the RMSE value for the second pair of models are different from the original work conducted by Salih Tuncer (https://github.com/SalihTuncer/AssessHumor), despite being identical, because we are using Tensorflow version 2.14.  So, the optimizers and functions and intermediate functions may behave a little different between versions.  Also, Tuncer did not set a seed before running his experiments, which hinders the reproducibility of his results. 

## Compute best model against test set

In [26]:
# Load the best peforming model
final_model = load_model('model_adam')

In [27]:
# get the test set
test_dataset = prep.get_test()
print(test_dataset)

<_BatchDataset element_spec=({'input_ids': TensorSpec(shape=(None, 256), dtype=tf.int32, name=None), 'attention_mask': TensorSpec(shape=(None, 256), dtype=tf.int32, name=None)}, TensorSpec(shape=(None,), dtype=tf.float32, name=None))>


In [28]:
# Evaluate the models against the validation set
generalization_error = final_model.evaluate(dev_dataset, return_dict=True)

ValueError: in user code:

    File "c:\Users\Paul_\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\engine\training.py", line 2042, in test_function  *
        return step_function(self, iterator)
    File "c:\Users\Paul_\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\engine\training.py", line 2025, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "c:\Users\Paul_\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\engine\training.py", line 2013, in run_step  **
        outputs = model.test_step(data)
    File "c:\Users\Paul_\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\engine\training.py", line 1893, in test_step
        y_pred = self(x, training=False)
    File "c:\Users\Paul_\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\utils\traceback_utils.py", line 70, in error_handler
        raise e.with_traceback(filtered_tb) from None

    

In [ ]:
# legacy
# preprocess the datasets and return a utility class which carries the datasets
max_seq_length = 256
batch_size = 16
prep = Preprocess(max_seq_length=max_seq_length, batch_size=batch_size)

# get the processed training and validation set
train_dataset = prep.get_train()
dev_dataset = prep.get_dev()

# create an albert model specialised on regression
model_rmsprop = NN().get_nn()
model_adam = NN().get_nn()

# use the RMSprop optimizer
optimizer_rmsprop = RMSprop(learning_rate = 1e-5, decay = 1e-8)
optimizer_adam = Adam(learning_rate = 1e-5, decay = 1e-8)

# compile the model with the appropriate optimizer, loss function, and evaluation metric
model_rmsprop.compile(optimizer=optimizer_rmsprop, loss=MeanSquaredError(),
            metrics=[RootMeanSquaredError('root_mean_squared_error')])
model_adam.compile(optimizer=optimizer_adam, loss=MeanSquaredError(),
            metrics=[RootMeanSquaredError('root_mean_squared_error')])

print("Fit model on training data.")

# train the model on the training set for 3 epochs
# and dusing the validation set for validation
history_rmsprop = model_rmsprop.fit(train_dataset, validation_data=dev_dataset, epochs=3)
print("RMSprop model successfully trained")

history_adam = model_adam.fit(train_dataset, validation_data=dev_dataset, epochs=3)
print("Adam model successfully trained")

print('Models successfully saved.')

# Evaluate the models against the validation set
results_rmsprop = model_rmsprop.evaluate(dev_dataset, return_dict=True)
results_adam = model_adam.evaluate(dev_dataset, return_dict=True)
# Print the RMSE for both models
print(f"RMSprop RMSE: {results_rmsprop['root_mean_squared_error']}")
print(f"Adam RMSE: {results_adam['root_mean_squared_error']}")

# output the model summary including the layers in the model,
# the output shape of each layer, and the number of parameters in each layer 
# print(model.summary())

# get the processed test set
# test_dataset = prep.get_test()

# print('Evaluate model with test data.')

# evaluate the test set
# print(model.evaluate(test_dataset, return_dict=True))

# print('Save the model weights.')

# save the model's weights
# model.save_weights('rmsprop/rmsprop')